# EEG-CNN → CLIP / DINOv2 (THINGS-EEG2, 17 каналів)

Легкий пайплайн під слабку GPU: усі дані лежать на GPU цілком (~60 МБ на суб'єкта), без DataLoader, модель мала.

**Рішення з EDA, які тут зашиті:**
- **17 posterior каналів** (файли `eeg/17ch`): top-1 identification 92% проти 55% на всіх 63.
- **Вікно 0–500 мс** (100 Гц → T = 50). Пізніші відліки лише додають шум.
- **Train: усереднення 4 повторів** кожного зображення. **Test: усереднення 80 повторів**, як у NICE.
- **Кліпінг ±10**: у train є викиди до ±300 (σ ≈ 0.8).
- **Val за концептами**: у train 1654 концепти × 10 зображень, у test концепти нові, тож val теж має бути по нових концептах.
- **InfoNCE з маскою дублікатів**: у LOSO те саме зображення приходить від 9 людей, тому дублікати в батчі інакше стають хибними негативами.

**Як запускати:** спочатку `MOCK = True` (синтетичні дані, ~1–2 хв), щоб перевірити, що все працює і accuracy ≫ 0.5%. Потім `MOCK = False`.

In [11]:
import os, sys, time, gc, copy, math, subprocess
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

MOCK = True                          # True = synthetic data (smoke test). False = real THINGS-EEG2
RUN_SUBJECTS = ['sub-01']
TARGETS  = ['clip', 'dinov2']
SETTINGS = ['within']                # 'within' та/або 'loso'
SEEDS    = [0]                       # фінально [0, 1, 2]

EEMB_DIR   = '/content/drive/MyDrive/Neural-Decoding/embeddings'
CACHE_DIR = '/content/drive/MyDrive/Neural-Decoding/eeg_cache'
RESULTS_CSV = '/content/drive/MyDrive/Neural-Decoding'

# --- дані
CROP_MS = (0, 500)                   # вікно в мс відносно появи стимулу
CLIP_ABS = 10.0                      # кліпінг викидів на рівні окремих повторів
VAL_FRAC_CONCEPTS = 0.05             # частка концептів train, яка йде у validation
SPLIT_SEED = 0                       # однаковий для всіх run, щоб split був спільний

# --- модель / тренування
F1, D, F2 = 16, 2, 32
K_T, K_S = 25, 7                     # ядра: часове (250 мс) та separable
POOL1, POOL2 = 2, 5
DROPOUT = 0.5
INIT_TEMP = 0.07
EPOCHS, PATIENCE = 40, 8
BATCH, LR, WD = 512, 1e-3, 1e-3

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| MOCK =', MOCK)

# ============================== константи датасету ==============================
ALL_SUBJECTS = [f'sub-{i:02d}' for i in range(1, 11)]
N_TRAIN, N_TEST, N_CH, N_CONCEPTS = 16540, 200, 17, 1654
TS = np.arange(100) * 0.01 - 0.2
CROP = (TS >= CROP_MS[0] / 1000 - 1e-9) & (TS < CROP_MS[1] / 1000 - 1e-9)
T = int(CROP.sum())
CONCEPT_ID = np.arange(N_TRAIN) // 10          # train відсортований по концептах, по 10 зображень (EDA)

# split train/val по концептах (спільний для всіх run)
_rng = np.random.default_rng(SPLIT_SEED)
VAL_CONCEPTS = _rng.choice(N_CONCEPTS, int(round(VAL_FRAC_CONCEPTS * N_CONCEPTS)), replace=False)
IS_VAL = np.isin(CONCEPT_ID, VAL_CONCEPTS)
print(f'T = {T} відліків | val: {len(VAL_CONCEPTS)} концептів = {IS_VAL.sum()} зображень | train: {(~IS_VAL).sum()}')

device: cuda | MOCK = True
T = 50 відліків | val: 83 концептів = 830 зображень | train: 15710


## 1. Дані: EEG та ембединги

In [12]:
REPO_ID, WORK = 'things-eeg2/things-eeg2-preprocessed', '/content/work'
_TOKEN = None
def get_token():
    global _TOKEN
    if _TOKEN: return _TOKEN
    try:
        from google.colab import userdata
        for k in ('hf_token', 'HF_TOKEN'):
            try: _TOKEN = userdata.get(k); break
            except Exception: pass
    except Exception: pass
    _TOKEN = _TOKEN or os.environ.get('hf_token') or os.environ.get('HF_TOKEN')
    if not _TOKEN:
        from getpass import getpass
        _TOKEN = getpass('Hugging Face token (read access): ')
    return _TOKEN

def _load_npy_dict(path):
    obj = np.load(path, allow_pickle=True)
    return obj.item() if isinstance(obj, np.ndarray) and obj.shape == () else obj

def load_subject_real(s):
    os.makedirs(CACHE_DIR, exist_ok=True); os.makedirs(WORK, exist_ok=True)
    cache = os.path.join(CACHE_DIR, f'{s}_17ch_{CROP_MS[0]}-{CROP_MS[1]}ms_clip{CLIP_ABS:g}.npz')
    if os.path.exists(cache):
        z = np.load(cache); return dict(train=z['train'], test=z['test'])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=False)
    from huggingface_hub import hf_hub_download
    out = {}
    for split, key, shape in [('test', 'test', (N_TEST, 80, N_CH, 100)), ('training', 'train', (N_TRAIN, 4, N_CH, 100))]:
        p = hf_hub_download(REPO_ID, f'eeg/17ch/{s}/preprocessed_eeg_{split}.npy', repo_type='dataset',
                            token=get_token(), local_dir=WORK)
        X = _load_npy_dict(p)['preprocessed_eeg_data']
        assert X.shape == shape, (s, split, X.shape)
        np.clip(X, -CLIP_ABS, CLIP_ABS, out=X)
        out[key] = X.mean(1)[:, :, CROP].astype(np.float32)
        del X; os.remove(p); gc.collect()
    np.savez(cache, **out)
    return out

FILE_PREFIX = {'clip': 'clip', 'dinov2': 'dino'}

def load_embeddings_real(target):
    p = FILE_PREFIX[target]
    tr = np.load(os.path.join(EMB_DIR, f'{p}_train.npy')).astype(np.float32)
    te = np.load(os.path.join(EMB_DIR, f'{p}_test.npy')).astype(np.float32)
    assert tr.shape[0] == N_TRAIN and te.shape[0] == N_TEST and tr.shape[1] == te.shape[1], (tr.shape, te.shape)
    return tr, te

# ============================== mock-дані ==============================
# EEG = лінійна функція прихованого вектора z + шум; ембединги = теж функція z. Тож зв'язок вчиться.
MOCK_DIMS = {'clip': 512, 'dinov2': 768}
_MOCK = {}
def _mock_init():
    if _MOCK: return
    rng = np.random.default_rng(123); k = 32
    _MOCK['z_tr'] = rng.standard_normal((N_TRAIN, k)).astype(np.float32)
    _MOCK['z_te'] = rng.standard_normal((N_TEST, k)).astype(np.float32)
    _MOCK['P'] = rng.standard_normal((k, N_CH * T)).astype(np.float32) / math.sqrt(k)
    for t, dim in MOCK_DIMS.items():
        M = rng.standard_normal((k, dim)).astype(np.float32)
        _MOCK[t] = tuple((z @ M + 0.5 * rng.standard_normal((len(z), dim))).astype(np.float32)
                         for z in (_MOCK['z_tr'], _MOCK['z_te']))

def load_subject_mock(s):
    _mock_init()
    rng = np.random.default_rng(int(s[-2:]))
    Ps = _MOCK['P'] + 0.5 * rng.standard_normal(_MOCK['P'].shape).astype(np.float32)
    mk = lambda z, noise: (z @ Ps + noise * rng.standard_normal((len(z), N_CH * T))).astype(np.float32).reshape(-1, N_CH, T)
    return dict(train=mk(_MOCK['z_tr'], 3.0), test=mk(_MOCK['z_te'], 1.0))

def load_embeddings_mock(target):
    _mock_init(); return _MOCK[target]

#кеш
_SUBJ, _EMB = {}, {}
def get_subject(s):
    if s not in _SUBJ:
        t0 = time.time(); _SUBJ[s] = load_subject_mock(s) if MOCK else load_subject_real(s)
        print(f'  loaded {s}: train {_SUBJ[s]["train"].shape}, test {_SUBJ[s]["test"].shape} ({time.time()-t0:.0f}s)')
    return _SUBJ[s]
def get_embeddings(target):
    if target not in _EMB: _EMB[target] = load_embeddings_mock(target) if MOCK else load_embeddings_real(target)
    return _EMB[target]

## 2. Модель та лосс (твоя частина з пропозиції)

In [13]:
class EEGEncoder(nn.Module):
    """EEGNet-style: (B, C, T) -> (B, emb_dim). Повертає НЕнормалізований вектор."""
    def __init__(self, n_channels, n_times, emb_dim, F1=16, D=2, F2=32, k_t=25, k_s=7,
                 pool1=2, pool2=5, dropout=0.5):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, k_t), padding=(0, k_t // 2), bias=False),          # temporal
            nn.BatchNorm2d(F1),
            nn.Conv2d(F1, F1 * D, (n_channels, 1), groups=F1, bias=False),          # depthwise spatial
            nn.BatchNorm2d(F1 * D), nn.ELU(), nn.AvgPool2d((1, pool1)), nn.Dropout(dropout))
        self.block2 = nn.Sequential(
            nn.Conv2d(F1 * D, F1 * D, (1, k_s), padding=(0, k_s // 2), groups=F1 * D, bias=False),  # separable
            nn.Conv2d(F1 * D, F2, 1, bias=False),
            nn.BatchNorm2d(F2), nn.ELU(), nn.AvgPool2d((1, pool2)), nn.Dropout(dropout))
        was_training = self.training; self.eval()
        with torch.no_grad(): n_feat = self.features(torch.zeros(2, n_channels, n_times)).shape[1]
        self.train(was_training)
        self.fc = nn.Linear(n_feat, emb_dim)

    def features(self, x):
        return self.block2(self.block1(x.unsqueeze(1))).flatten(1)

    def forward(self, x):
        return self.fc(self.features(x))


class InfoNCELoss(nn.Module):
    """Симетричний InfoNCE з learnable температурою. img_ids маскує дублікати зображень у батчі."""
    def __init__(self, init_temp=0.07, symmetric=True):
        super().__init__()
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1 / init_temp), dtype=torch.float32))
        self.symmetric = symmetric

    def forward(self, eeg_emb, img_emb, img_ids=None):
        e, v = F.normalize(eeg_emb, dim=-1), F.normalize(img_emb, dim=-1)
        logits = self.logit_scale.clamp(max=math.log(100.0)).exp() * e @ v.T
        if img_ids is not None:
            dup = (img_ids[:, None] == img_ids[None, :]) & ~torch.eye(len(e), dtype=torch.bool, device=e.device)
            logits = logits.masked_fill(dup, -1e4)
        y = torch.arange(len(e), device=e.device)
        loss = F.cross_entropy(logits, y)
        if self.symmetric: loss = (loss + F.cross_entropy(logits.T, y)) / 2
        return loss

In [14]:
# ---- швидкі sanity-перевірки (до тренування) ----
m = EEGEncoder(N_CH, T, 512, F1, D, F2, K_T, K_S, POOL1, POOL2, DROPOUT)
x = torch.randn(256, N_CH, T)
out = m(x)
print('out shape:', tuple(out.shape), '| params:', sum(p.numel() for p in m.parameters()))
crit = InfoNCELoss(INIT_TEMP)
loss0 = crit(out, torch.randn(256, 512), torch.arange(256))
print(f'initial loss {loss0.item():.3f}  (ln(256) = {math.log(256):.3f}, на старті має бути близько)')
# дублікати: якщо всі ids однакові, всі недіагональні логіти маскуються, лосс -> ~0
print('loss з усіма однаковими ids:', round(crit(out, torch.randn(256, 512), torch.zeros(256, dtype=torch.long)).item(), 4))
del m, out, crit

out shape: (256, 512) | params: 84784
initial loss 5.790  (ln(256) = 5.545, на старті має бути близько)
loss з усіма однаковими ids: 0.0


## 3. Evaluate і тренування

In [15]:
@torch.no_grad()
def embed(model, X, bs=2048):
    model.eval()
    return torch.cat([model(X[i:i + bs]) for i in range(0, len(X), bs)])

def retrieval(eeg_emb, img_emb):
    """len(eeg_emb) == len(img_emb); i-й EEG відповідає i-му зображенню. Повертає (top1, top5)."""
    S = F.normalize(eeg_emb, dim=-1) @ F.normalize(img_emb, dim=-1).T
    top = S.topk(5, dim=1).indices
    gt = torch.arange(len(S), device=S.device)[:, None]
    return (top[:, :1] == gt).float().mean().item(), (top == gt).any(1).float().mean().item()

def evaluate(model, X_test, img_emb_test):
    """200-way retrieval, косинус. X_test (200, C, T) вже нормалізований, img_emb_test (200, D)."""
    return retrieval(embed(model, X_test), img_emb_test)

def val_score(model, Xv, Ev, chunks):
    """Retrieval усередині блоків по 200 (як у test: 200-way, chance = 0.5%). Усереднення по блоках."""
    e = embed(model, Xv)
    r = [retrieval(e[c], Ev[c]) for c in chunks]
    return float(np.mean([a for a, _ in r])), float(np.mean([b for _, b in r]))

def train_one(Xtr, ids_tr, Xv, Ev, E_all, seed):
    """Xtr (N, C, T), ids_tr (N,) -> індекс зображення, E_all (16540, D) на GPU. Повертає (best_state, history)."""
    torch.manual_seed(seed); np.random.seed(seed)
    emb_dim = E_all.shape[1]
    model = EEGEncoder(N_CH, T, emb_dim, F1, D, F2, K_T, K_S, POOL1, POOL2, DROPOUT).to(DEVICE)
    crit = InfoNCELoss(INIT_TEMP).to(DEVICE)
    opt = torch.optim.AdamW(list(model.parameters()) + list(crit.parameters()), lr=LR, weight_decay=WD)
    N = len(Xtr); steps = math.ceil(N / BATCH)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS * steps)
    g = torch.Generator(device='cpu').manual_seed(seed + 1000)
    n_chunks = len(Xv) // 200
    vperm = torch.randperm(len(Xv), generator=torch.Generator().manual_seed(SPLIT_SEED))[:n_chunks * 200].view(n_chunks, 200).to(DEVICE)
    best, best_state, bad, hist = -1, None, 0, []
    for ep in range(1, EPOCHS + 1):
        model.train(); perm = torch.randperm(N, generator=g).to(DEVICE); tot = 0.0; nb = 0
        for i in range(0, N, BATCH):
            idx = perm[i:i + BATCH]
            if len(idx) < 32: continue
            ids = ids_tr[idx]
            loss = crit(model(Xtr[idx]), E_all[ids], ids)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
            tot += loss.item(); nb += 1
        v1, v5 = val_score(model, Xv, Ev, vperm)
        hist.append(dict(epoch=ep, loss=tot / nb, val_top1=v1, val_top5=v5, best=False,
                         temp=1 / crit.logit_scale.clamp(max=math.log(100.0)).exp().item()))
        if v1 > best:
            best, bad, best_state = v1, 0, copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})
            hist[-1]['best'] = True
        else:
            bad += 1
        if ep % 5 == 0 or ep == 1:
            h = hist[-1]; print(f'    ep {ep:2d} | loss {h["loss"]:.3f} | val top1 {100*v1:5.1f}% top5 {100*v5:5.1f}% | temp {h["temp"]:.3f}')
        if bad >= PATIENCE:
            print(f'    early stop at epoch {ep}'); break
    model.load_state_dict(best_state)
    return model, pd.DataFrame(hist)

## 4. Один run (target × setting × subject × seed)

In [16]:
def make_split(setting, s):
    """Повертає numpy: Xtr, ids_tr, Xv, ids_v, Xte. Для LOSO val береться лише з 9 тренувальних людей."""
    if setting == 'within':
        d = get_subject(s)
        X, ids = d['train'], np.arange(N_TRAIN)
    else:
        others = [o for o in ALL_SUBJECTS if o != s]
        X = np.concatenate([get_subject(o)['train'] for o in others])
        ids = np.tile(np.arange(N_TRAIN), len(others))
    val_pos = np.where(IS_VAL[ids])[0]
    val_pos = np.random.default_rng(SPLIT_SEED).permutation(val_pos)
    _, first = np.unique(ids[val_pos], return_index=True)       # одне EEG на кожне val-зображення (у LOSO випадкова людина)
    val_pos = val_pos[first]
    tr_pos = np.where(~IS_VAL[ids])[0]
    return X[tr_pos], ids[tr_pos], X[val_pos], ids[val_pos], get_subject(s)['test']

def run_one(target, setting, s, seed):
    t0 = time.time()
    emb_tr, emb_te = get_embeddings(target)
    Xtr, ids_tr, Xv, ids_v, Xte = make_split(setting, s)
    to = lambda a, dt=torch.float32: torch.as_tensor(a, dtype=dt, device=DEVICE)
    Xtr, Xv, Xte = to(Xtr), to(Xv), to(Xte)
    mu = Xtr.mean(dim=(0, 2), keepdim=True); sd = Xtr.std(dim=(0, 2), keepdim=True) + 1e-6   # статистика тільки з train
    Xtr, Xv, Xte = (Xtr - mu) / sd, (Xv - mu) / sd, (Xte - mu) / sd
    E_all, E_te = to(emb_tr), to(emb_te)
    ids_tr_t, ids_v_t = to(ids_tr, torch.long), to(ids_v, torch.long)
    model, hist = train_one(Xtr, ids_tr_t, Xv, E_all[ids_v_t], E_all, seed)
    top1, top5 = evaluate(model, Xte, E_te)
    best_ep = int(hist.loc[hist.best, 'epoch'].iloc[-1])
    row = dict(target=target, setting=setting, subject=s, seed=seed, top1=top1, top5=top5,
               val_top1=float(hist.val_top1.max()), best_epoch=best_ep, epochs_run=len(hist),
               n_train=len(Xtr), time_s=round(time.time() - t0, 1))
    del Xtr, Xv, Xte, E_all, E_te, model
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    gc.collect()
    return row, hist

## 5. Запуск усіх комбінацій

Результати дописуються в `RESULTS_CSV` після кожного run. Якщо Colab обірветься, просто запусти клітинку знову: готові комбінації пропускаються.

In [17]:
done = set()
if os.path.exists(RESULTS_CSV):
    old = pd.read_csv(RESULTS_CSV)
    done = set(zip(old.target, old.setting, old.subject, old.seed))
    print(f'знайдено {len(done)} готових run, їх пропускаємо')

last_hist = None
for setting in SETTINGS:
    for s in RUN_SUBJECTS:
        for target in TARGETS:
            for seed in SEEDS:
                if (target, setting, s, seed) in done: continue
                print(f'>> {setting} | {s} | {target} | seed {seed}')
                row, last_hist = run_one(target, setting, s, seed)
                print(f'   TEST top1 {100*row["top1"]:.1f}%  top5 {100*row["top5"]:.1f}%  '
                      f'(val top1 {100*row["val_top1"]:.1f}%, best epoch {row["best_epoch"]}, {row["time_s"]}s)')
                pd.DataFrame([row]).to_csv(RESULTS_CSV, mode='a', header=not os.path.exists(RESULTS_CSV), index=False)
print('готово')

>> within | sub-01 | clip | seed 0
  loaded sub-01: train (16540, 17, 50), test (200, 17, 50) (0s)
    ep  1 | loss 4.905 | val top1  91.9% top5  99.9% | temp 0.069
    ep  5 | loss 0.946 | val top1  99.7% top5 100.0% | temp 0.060
    ep 10 | loss 0.510 | val top1 100.0% top5 100.0% | temp 0.054
    ep 15 | loss 0.376 | val top1 100.0% top5 100.0% | temp 0.050
    early stop at epoch 15
   TEST top1 100.0%  top5 100.0%  (val top1 100.0%, best epoch 7, 4.5s)


OSError: Cannot save file into a non-existent directory: '/content/drive/MyDrive'

In [ ]:
# ---- підсумок: спочатку усереднення по seed для кожного суб'єкта, потім mean±std по суб'єктах ----
res = pd.read_csv(RESULTS_CSV)
per_subj = res.groupby(['target', 'setting', 'subject'])[['top1', 'top5']].mean()
g = per_subj.groupby(['target', 'setting'])
summary = pd.concat({'top1_mean%': g.top1.mean() * 100, 'top1_std%': g.top1.std() * 100,
                     'top5_mean%': g.top5.mean() * 100, 'top5_std%': g.top5.std() * 100,
                     'n_subjects': g.size()}, axis=1).round(1)
display(summary)
print('chance: top1 0.5% | top5 2.5%   ·   NICE (reference): top1 15.6% | top5 42.8%')

if last_hist is not None:
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
    ax[0].plot(last_hist.epoch, last_hist.loss); ax[0].set_title('train loss (останній run)'); ax[0].set_xlabel('epoch')
    ax[1].plot(last_hist.epoch, 100 * last_hist.val_top1, label='val top-1'); ax[1].plot(last_hist.epoch, 100 * last_hist.val_top5, label='val top-5')
    ax[1].axhline(0.5, c='k', ls=':', lw=.8); ax[1].set_title('validation (200-way, %)'); ax[1].set_xlabel('epoch'); ax[1].legend()
    plt.tight_layout(); plt.show()